# 04 • CNN : classer des images et examiner les erreurs

`[MÉTA | Formation 4-024 | Niveau Application | TP 04 | Mode CPU local]`

**Objectif :** Comparer une architecture dense et un CNN sur les mêmes images.

**Temps indicatif :** 70 min. Ces temps sont répartis dans le conducteur, pas additionnés hors des 18 heures.

**Prérequis :** TP 01, notions de convolution.

**Preuves de réussite :** Forme N×1×8×8, paramètres comptés, courbes et erreurs, rechargement.

**Sources :** R02, R05 ; digits scikit-learn.

Les jeux métier sont synthétiques. Aucun fichier personnel ou fiscal réel ne doit être chargé. Les résultats obtenus ici ne constituent pas une validation métier.

**Mode d’emploi :** exécuter les cellules dans l’ordre. Les cellules d’exercice du cahier apprenant sont à compléter ; le corrigé contient le code et des résultats de référence sur CPU.

In [ ]:
from pathlib import Path
import sys, os, json
# Chercher le kit depuis le répertoire du notebook ou celui de lancement.
HERE = Path.cwd().resolve()
TP_ROOT = next((p for p in [HERE, *HERE.parents] if (p / "modules" / "atelier.py").exists()), None)
if TP_ROOT is None:
    raise FileNotFoundError("Ouvrir ce notebook depuis le dossier 03_Travaux_pratiques du kit décompressé.")
sys.path.insert(0, str(TP_ROOT / "modules"))
os.environ.setdefault("KERAS_BACKEND", "torch")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from torch import nn
from atelier import *
seed_all(42)
print("Moteur disponible :", torch.__version__, "| Données :", DATA)


## 1. Images et split
Les images publiques contiennent des chiffres manuscrits 8 × 8, pas des objets fonciers. L’échelle 0 à 16 est une propriété documentée du jeu. On divise par 16 sans apprendre une statistique sur le test.

In [ ]:
d=digits_data();X,y=d['train'];VX,Vy=d['validation']
assert X.shape[1:]==(1,8,8)
print('Formes :',X.shape,VX.shape,'Valeurs :',X.min(),X.max())
fig,ax=plt.subplots(figsize=(4,4));ax.imshow(X[0,0],cmap='gray');ax.set_title(f'Exemple, classe {y[0]}');ax.axis('off');plt.show()

## 2. MLP et CNN
MLP : aplatir 64 pixels, couche 64, sortie 10. CNN : deux convolutions 3 × 3 avec padding 1, deux poolings 2 × 2, puis 64 caractéristiques et 10 logits. Les filtres sont appris, et non dessinés à la main.

In [ ]:
# EXERCICE À COMPLÉTER
# Conserver des logits à la sortie. Vérifier les dimensions après chaque pooling.
# La correction est fournie séparément au formateur.
raise NotImplementedError("Compléter cette cellule puis relancer avant de poursuivre.")

## 3. Entraînement comparé
La boucle fit_model est dans modules/atelier.py et peut être ouverte pour inspection. Elle utilise CrossEntropyLoss, Adam et une validation sans gradient. Le temps mural n’est qu’un repère local, pas un benchmark matériel.

In [ ]:
import time
comparatif={}
for nom,modele in [('Dense',dense),('CNN',cnn)]:
    debut=time.perf_counter();h=fit_model(modele,d['train'],d['validation'],task='multi',epochs=15)
    pred=predict(modele,VX,'multi').argmax(1)
    comparatif[nom]={'exactitude':float(accuracy_score(Vy,pred)),'f1_macro':float(f1_score(Vy,pred,average='macro')),'secondes':time.perf_counter()-debut}
    plot_history(h,nom+' : entraînement et validation','04_'+nom.lower()+'.png')
print(pd.DataFrame(comparatif).T)

## 4. Galerie d’erreurs
Le score maximum du softmax est une confiance du modèle, non une garantie. Pour chaque erreur : ambiguïté, classe proche, qualité faible, éventuelle étiquette à contrôler. Ne pas prétendre modifier le test pour supprimer les erreurs.

In [ ]:
scores=predict(cnn,VX,'multi');pred=scores.argmax(1);erreurs=np.flatnonzero(pred!=Vy)
print('Erreurs sur validation :',len(erreurs))
# Une seule figure, montage manuel d'images, sans sous-graphiques.
selection=erreurs[:12];canvas=np.ones((len(selection)*10 if len(selection) else 10,8))
for j,ix in enumerate(selection):canvas[j*10:j*10+8]=VX[ix,0]
fig,ax=plt.subplots(figsize=(5,max(3,len(selection)*.55)))
ax.imshow(canvas,cmap='gray');ax.axis('off')
for j,ix in enumerate(selection):ax.text(9,j*10+4,f'Vrai {Vy[ix]} / prédit {pred[ix]} / score {scores[ix,pred[ix]]:.2f}',va='center',fontsize=10)
ax.set_xlim(0,39);fig.tight_layout();fig.savefig(RESULTS/'04_erreurs.png',dpi=150)
print(pd.DataFrame({'indice':selection,'vrai':Vy[selection],'predit':pred[selection]}))

## 5. Sauvegarde et livraison
Tester le nouveau modèle rechargé et documenter le format d’entrée. Le test n’est pas encore utilisé pour choisir des paramètres.

In [ ]:
# EXERCICE À COMPLÉTER
# Créer un nouvel objet, charger state_dict avec weights_only=True puis comparer les sorties en mode évaluation.
# La correction est fournie séparément au formateur.
raise NotImplementedError("Compléter cette cellule puis relancer avant de poursuivre.")

## 6. Prolongements
Modifier une seule variable : filtres, dropout ou durée. Refaire une hypothèse écrite. Une meilleure validation dans un essai ne suffit pas à affirmer une meilleure généralisation dans tous les contextes. **Secours CPU :** 6 époques et analyse des courbes ; les résultats de référence restent disponibles au formateur.